In [0]:
import json
import urllib.request
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.spb")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.spb.landing")

LANDING_DIR = "/Volumes/workspace/spb/landing"
CITY_ID = "ff3a1bef-1d4a-4f98-8221-19c888ad76f1"  # https://explore.overturemaps.org

latest = json.load(urllib.request.urlopen("https://stac.overturemaps.org/catalog.json"))["latest"]
base = f"s3://overturemaps-us-west-2/release/{latest}"


city = (
    spark.read.parquet(f"{base}/theme=divisions/type=division_area/")
    .where(f"division_id = '{CITY_ID}' AND class = 'land'")
    .select("id", 
            "division_id",
            "subtype", 
            "class",
            "bbox",
            "geometry",
            F.col("names.primary").alias("name"))
)

row = city.select("bbox.*").first()
BBOX = {"xmin": row.xmin,
        "xmax": row.xmax,
        "ymin": row.ymin,
        "ymax": row.ymax}

where = f"""
    bbox.xmin BETWEEN {BBOX['xmin']} AND {BBOX['xmax']}
    AND bbox.ymin BETWEEN {BBOX['ymin']} AND {BBOX['ymax']}
"""

city.write.mode("overwrite").parquet(f"{LANDING_DIR}/division_area")

(
    spark.read.parquet(f"{base}/theme=places/type=place/")
    .where(where)
    .select("id", "taxonomy", "bbox", F.col("names.primary").alias("name"),
            "basic_category", "confidence")
    .write.mode("overwrite").parquet(f"{LANDING_DIR}/places")
)

(
    spark.read.parquet(f"{base}/theme=buildings/type=building/")
    .where(where)
    .select("id", "geometry", "bbox", F.col("names.primary").alias("name"),
            "subtype", F.col("class").alias("building_class"))
    .write.mode("overwrite").parquet(f"{LANDING_DIR}/buildings")
)

In [0]:
division_area = (
    spark.read.parquet(f"{base}/theme=divisions/type=division_area/")
    .where("division_id = 'ff3a1bef-1d4a-4f98-8221-19c888ad76f1'")
    .select("id", 
            "bbox",
            "names"
    )
)

In [0]:
division_area.select("names.primary", "bbox.xmin", "bbox.xmax", "bbox.ymin", "bbox.ymax").show(truncate=False)

In [0]:
connectors = spark.read.parquet(f"{base}/theme=transportation/type=connector/")
connectors.printSchema()
connectors.select("id", "bbox").show(truncate=False)

segments = spark.read.parquet(f"{base}/theme=transportation/type=segment/")
segments.printSchema()
segments.select("id","names.primary", "bbox","subtype","connectors").show(truncate=False)

In [0]:
pip install osmnx

In [0]:
%restart_python

In [0]:
import osmnx as ox
from shapely.geometry import Polygon

polygon = Polygon([(BBOX['xmin'], BBOX['ymin']), (BBOX['xmin'], BBOX['ymax']), (BBOX['xmax'], BBOX['ymax']), (BBOX['xmax'], BBOX['ymin'])])
G = ox.graph_from_polygon(polygon, network_type='walk')

fig,ax = ox.plot_graph(G)

In [0]:
print("raw graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")|

In [0]:
import networkx as nx
U = ox.convert.to_undirected(G)
components = sorted(nx.connected_components(U), key=len, reverse=True)
print("component sizes (top 5):", [len(c) for c in components[:5]])

In [0]:
U = U.subgraph(components[0]).copy()
print("kept:", U.number_of_nodes(), "nodes,", U.number_of_edges(), "edges")

In [0]:
nodes, edges = ox.convert.graph_to_gdfs(U)

nodes_out = (
    nodes.reset_index()[["osmid", "x", "y"]]
    .rename(columns={"osmid": "node_id", "x": "lon", "y": "lat"})
)

edges_out = (
    edges.reset_index()[["u", "v", "length"]]
    .rename(columns={"u": "src", "v": "dst", "length": "length_m"})
    .groupby(["src", "dst"], as_index=False)["length_m"].min()   # keep shortest parallel edge
)

In [0]:
print("edge length min/max (m):", edges_out.length_m.min(), edges_out.length_m.max())
print("total network length (km):", round(edges_out.length_m.sum() / 1000))

In [0]:
print(edges_out.nlargest(10, "length_m"))
print((edges_out.length_m < 1).sum(), "edges under 1 m")

In [0]:
import pandas as pd

top = edges_out.nlargest(3, "length_m")
ids = pd.concat([top.src, top.dst])
print(nodes_out[nodes_out.node_id.isin(ids)])

In [0]:
nodes_out.to_parquet("walk_nodes.parquet", index=False)
edges_out.to_parquet("walk_edges.parquet", index=False)

spark.createDataFrame(nodes_out).write.mode("overwrite").parquet(f"{LANDING_DIR}/walk_nodes")
spark.createDataFrame(edges_out).write.mode("overwrite").parquet(f"{LANDING_DIR}/walk_edges")